# DataBricks と Numpy の相性は最悪に近い
- DataBricks と Numpy の相性は最悪に近い(Repeat!!!)
- Kernel Crush が発生しやすく、回避が困難である
- そのため、Deltalake TableおよびSparkを用いた構成に書き換えることとした

In [ ]:
%pip install                                              \
        numpy                 adlfs               httpx   \
        python-dotenv         openai              polars  \
		azure-ai-inference    packaging

%restart_python

In [ ]:
import os
import gc
import ast
import asyncio
import json
import httpx
import numpy  as np
import scipy  as sp
from tqdm         import tqdm
from tqdm.asyncio import tqdm as asynctqdm
from dotenv       import load_dotenv
from openai       import AsyncOpenAI

import pandas as pd
from pyspark.sql import SparkSession, Window, types
from pyspark.sql.functions import col
import pyspark.sql.functions as F
from delta       import configure_spark_with_delta_pip

from azure.ai.inference.models import SystemMessage, UserMessage

from llm_agent   import LlmAgent


In [ ]:
builder = SparkSession.builder\
            .config("spark.sql.sources.commitProtocolClass", "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol")\
            .config("spark.sql.parquet.output.committer.class", "org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter")\
            .config("spark.mapreduce.fileoutputcommitter.marksuccessfuljobs","false")\
            .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")\
            .config("spark.sql.adaptive.enabled", True)\
            .config("spark.sql.shuffle.partitions", "auto")\
            .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", "100MB")\
            .config("spark.sql.adaptive.coalescePartitions.enabled", True)\
            .config("spark.sql.dynamicPartitionPruning.enabled", True)\
            .config("spark.sql.autoBroadcastJoinThreshold", "10MB")\
            .config("spark.sql.session.timeZone", "Asia/Tokyo")\
            .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
            .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")\
            .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
            .config("spark.databricks.delta.write.isolationLevel", "SnapshotIsolation")\
            .config("spark.databricks.delta.optimizeWrite.enabled", True)\
            .config("spark.databricks.delta.autoCompact.enabled", True)
            # Delta Lake 用の SQL コミットプロトコルを指定
            # Parquet 出力時のコミッタークラスを指定
            # Azure Blob Storage (ABFS) 用のコミッターファクトリを指定
            # '_SUCCESS'で始まるファイルを書き込まないように設定
            # JavaSerializerからKryoSerializerへの変更
            # AQE(Adaptive Query Execution)の有効化
            # パーティション数を自動で調整するように設定
            # シャッフル後の1パーティションあたりの最小サイズを指定
            # AQEのパーティション合成の有効化
            # 動的パーティションプルーニングの有効化
            # 小さいテーブルのブロードキャスト結合への自動変換をするための閾値調整
            # SparkSessionのタイムゾーンを日本標準時刻に設定
            # Apache Arrow 形式で Pandas<=>Spark 変換を行う
            # Delta Lake固有のSQL構文や解析機能を拡張モジュールとして有効化
            # SparkカタログをDeltaLakeカタログへ変更
            # Delta Lake書き込み時のアイソレーションレベルを「スナップショット分離」に設定
            # 書き込み時にデータシャッフルを行い、大きなファイルを生成する機能の有効化
            # 書き込み後に小さなファイルを自動で統合する機能の有効化

# 追加の設定処理
# クラスターサイズによって、動的に書き換えること
builder = builder\
			.config("spark.driver.memory", "64g")\
    		.config("spark.driver.maxResultSize", "32g")\
			.config("spark.rpc.message.maxSize", "1024")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# メモリ管理
del builder
gc.collect()


In [ ]:
# .env ファイルを読み込む
load_dotenv()

# 環境変数の取得
AI_FOUNDRY_ENDPOINT       = os.environ.get("AI_FOUNDRY_ENDPOINT")
AI_FOUNDRY_API_KEY        = os.environ.get("AI_FOUNDRY_API_KEY")
AI_FOUNDRY_MODEL          = os.environ.get("AI_FOUNDRY_MODEL")
LLM_MAX_TOKENS            = os.environ.get("LLM_MAX_TOKENS")
LLM_TEMPERATURE           = os.environ.get("LLM_TEMPERATURE")
LLM_TOP_P                 = os.environ.get("LLM_TOP_P")

# ウェジットからの設定の読み込み
BRAND_NAME                = dbutils.widgets.get('BRAND_NAME')
TARGET_BRAND_WORDS        = dbutils.widgets.get('TARGET_BRAND_WORDS')
EXTRACTION_ID             = dbutils.widgets.get('EXTRACTION_ID')

# メモ：
# 環境変数・ウィジット経由でのパラメータの取得方法では、無条件にパラメータの型がStringに変換されてしまう
# そのため、受け取ったパラメータを適切に型変換する必要がある
LLM_MAX_TOKENS       = int(LLM_MAX_TOKENS)
LLM_TEMPERATURE      = float(LLM_TEMPERATURE)
LLM_TOP_P            = float(LLM_TOP_P)
EXTRACTION_ID        = int(EXTRACTION_ID)

# メモ：
# TARGET_BRAND_WORDS の中身として
# "['aaa', 'bbb', 'ccc']"
# "aaa" or "'aaaaa'"
# 上記のような「文字列」を想定している
# 
# このような文字列を正しくパースするために、astライブラリを利用することとした
try:
	TARGET_BRAND_WORDS = ast.literal_eval(TARGET_BRAND_WORDS)
	if isinstance(TARGET_BRAND_WORDS, str):
		TARGET_BRAND_WORDS = [TARGET_BRAND_WORDS]
except (ValueError, SyntaxError):
	TARGET_BRAND_WORDS = [TARGET_BRAND_WORDS]

# 簡易デバッグ用
# print(f'AI_FOUNDRY_ENDPOINT:       {AI_FOUNDRY_ENDPOINT}')       # セキュリティリスクのため、コメントアウト
# print(f'AI_FOUNDRY_API_KEY:        {AI_FOUNDRY_API_KEY}')        # セキュリティリスクのため、コメントアウト
print(f'AI_FOUNDRY_MODEL:          {AI_FOUNDRY_MODEL}')
print(f'LLM_MAX_TOKENS:            {LLM_MAX_TOKENS}')
print(f'LLM_TEMPERATURE:           {LLM_TEMPERATURE}')
print(f'LLM_TOP_P:                 {LLM_TOP_P}')
print(f'EXTRACTION_ID:             {EXTRACTION_ID}')

In [ ]:
limits      = httpx.Limits(max_keepalive_connections=20, max_connections=300)
timeout     = httpx.Timeout(300.0, connect=5.0)
http_client = httpx.AsyncClient(limits=limits, timeout=timeout)
openai_cli  = AsyncOpenAI(
                    base_url=AI_FOUNDRY_ENDPOINT,
                    api_key=AI_FOUNDRY_API_KEY,
                    http_client=http_client,
                    max_retries=3
                )
semaphore   = asyncio.Semaphore(50)
llmClient   = LlmAgent(openai_cli, AI_FOUNDRY_MODEL, int(LLM_MAX_TOKENS), float(LLM_TEMPERATURE), float(LLM_TOP_P), semaphore)



In [ ]:
output_file_path = f'ブランドLP/{BRAND_NAME}.md'

# 商品分析情報が存在しない場合
if not os.path.exists(output_file_path):
	with open(f'ブランドLP/商品分析.md', 'r', encoding='utf-8') as f:
		sys_prompt = f.read()
		sys_prompt = sys_prompt.replace('【WORD_LIST】', ', '.join(TARGET_BRAND_WORDS))

	messages  = []
	messages.append(SystemMessage(content=sys_prompt))
	product_analysis = await llmClient.complete(messages)
	with open(output_file_path, 'w', encoding='utf-8') as f:
		f.write(json.dumps(product_analysis, indent=4, ensure_ascii=False))

# 商品分析情報の読み込み
with open(output_file_path, 'r', encoding='utf-8') as f:
    product_analysis = f.read()
product_analysis

In [ ]:
# 特定の地点に対するADIDリストを取得
POLYGONLIST_TABLE = "adinte_adintedmp.envprod.polygonlist"
sdf_polygonlist   = spark.read.table(POLYGONLIST_TABLE)\
							.select(['project', 'caption', 'extraction_id', 'branch_id', 'spot_id'])\
                            .filter(col('extraction_id') == EXTRACTION_ID)
ADIDLIST_TABLE    = "adinte_adintedmp.envprod.source"
sdf_adidlist      = spark.read.table(ADIDLIST_TABLE)\
							.select(['extraction_id', 'branch_id', 'spot_id', 'adid'])\
                            .filter(col('extraction_id') == EXTRACTION_ID)\
							.join(sdf_polygonlist, on=['extraction_id', 'branch_id', 'spot_id'], how='inner')\
                            .select(['caption', 'adid'])\
                            .dropDuplicates()
target_adidlist   = [row['adid'] for row in sdf_adidlist.select('adid').dropDuplicates().toLocalIterator()]

# メモリ管理
sdf_polygonlist.unpersist()
del sdf_polygonlist
spark.catalog.clearCache()
gc.collect()

sdf_adidlist.display()

In [ ]:
# メモ：
# cohort.npzは以下のようなデータ構成になっている
# cohort.npz
#     |-- data              : 計算済みコホート係数行列
#     |-- indices           : データの位置指定子(列)
#     |-- indptr            : 行ごとのスライスインデックス
#     |-- shape             : コホート係数行列の形(ADIDのリスト × コホートキャプションのリスト)
#     |-- adid_list         : ADIDのリスト(列)
#     |-- business_codelist : コホートキャプションIDのリスト(行)
TARGET      = "/Volumes/stgadintedmpadintedi/featurestore/behaviorvector/cohort.npz"
npz         = np.load(TARGET, allow_pickle=True)
np_cohort   = sp.sparse.csr_matrix((npz["data"], npz["indices"], npz["indptr"]), shape=tuple(npz["shape"]))
np_adidlist = npz["adid_list"]

# 特定のADIDのみの抜き出し
indexer     = pd.Index(np_adidlist)
indices     = indexer.get_indexer(target_adidlist)
indices     = indices[indices != -1]
np_cohort   = np_cohort[indices, :]
np_adidlist = np_adidlist[indices]


# メモ：
# L2正規化を行うにあたって、疎行列専用に書く必要がある
# 密行列であれば簡単に書くことが可能であるが、今回の要件に適合しない
# そのため疎行列な対角行列を経由することにより、これを実現することとした
# 
# 当初すでにL2正規化は適用済みであったが、時々適用されていない状態になることがあるようだ
# そのため、改めて適用することとした
# L2正規化
if not np.allclose(np_cohort.power(2).sum(axis=1), 1.0):
	l2norm_vector = sp.sparse.linalg.norm(np_cohort, axis=1)
	np_cohort     = sp.sparse.diags(1 / np.maximum(l2norm_vector, 1e-10)) @ np_cohort


# メモリ管理
del target_adidlist
del npz, indexer, indices
del l2norm_vector
gc.collect()


# CODELISTに対応するキャプションの文脈行列を取得
CAPTION_CONTEXT_MATRIX = "cohort_caption_matrix.npz"
npz                    = np.load(CAPTION_CONTEXT_MATRIX, allow_pickle=True)
relational_spots       = npz["business_placelist"]

# Sparkフレームワークに変換する
coo_cohort   = np_cohort.tocoo()
joined_spots = np.array(['_'.join(spot) for spot in relational_spots])
pdf_cohort   = pd.DataFrame({
						'adid'           : np_adidlist[ coo_cohort.row], 
						'cohort_caption' : joined_spots[coo_cohort.col], 
						'value'          : coo_cohort.data
					})
sdf_cohort   = spark.createDataFrame(pdf_cohort)

# メモリ管理
del np_cohort, np_adidlist
del npz, relational_spots
del coo_cohort, joined_spots, pdf_cohort
gc.collect()

sdf_cohort.display()

In [ ]:
# 突貫のため、精度無視
# 突貫のため、精度無視
# 突貫のため、精度無視
# 突貫のため、精度無視
# 突貫のため、精度無視


# 特定のADIDに対する性別・年齢層を取得
window_spec      = Window.partitionBy('caption')
AGE_GENDER_TABLE = "adinte_datainfrastructure.master.mobilewalla_agegender"
sdf_agegender    = spark.read.table(AGE_GENDER_TABLE)\
						.select( ['adid', 'age', 'gender'])\
                        .join(sdf_adidlist, on='adid', how='inner')\
                        .select( ['caption', 'adid', 'age', 'gender'])\
                        .groupBy(['caption', 'age', 'gender'])\
                        .agg(F.count('*').alias('count'))\
                        .withColumn('probability', F.col('count') / F.sum('count').over(window_spec))\
                        .select( ['caption', 'age', 'gender', 'probability'])\
                        .withColumnRenamed('caption', 'target')\
                        .select( ['target', 'age', 'gender', 'probability'])


# メモ：
# np_cohort 行列はL2正規化を施している
# そのため、その要素の2乗和は1になる
# これを確率として解釈することで、統計的な枠組みで処理を行うことが狙いである
# 
# 特定のADIDに対する行動ベクトルを取得
window_spec      = Window.partitionBy('target')
sdf_behavioral   = sdf_cohort\
						.withColumn('probability', F.pow(col('value'), 2))\
                        .select(['adid', 'cohort_caption', 'probability'])\
                        .join(sdf_adidlist, on='adid', how='inner')\
                        .select(['caption', 'adid', 'cohort_caption', 'probability'])\
                        .withColumnRenamed('caption', 'target')\
                        .select( ['target', 'adid', 'cohort_caption', 'probability'])\
                        .groupBy(['target', 'cohort_caption'])\
                        .agg(F.sum('probability').alias('sum_probability'))\
                        .withColumn('probability', col('sum_probability') / F.sum('sum_probability').over(window_spec))\
                        .select( ['target', 'cohort_caption', 'probability'])

# メモリ管理
sdf_adidlist.unpersist()
sdf_cohort.unpersist()
del sdf_adidlist, sdf_cohort
spark.catalog.clearCache()
gc.collect()

sdf_behavioral.display()

In [ ]:
# LLM用プロンプトの作成のための前処理
sdf_agegender_prep  = sdf_agegender\
						.withColumn('str_agegender', F.format_string("'%s' × '%s'", F.col("age"), F.col("gender")))\
						.withColumn('str_agegender', F.format_string('(%s, %s)', F.col('str_agegender'), F.col('probability')))\
						.select(['target', 'str_agegender'])\
						.groupBy('target')\
						.agg(F.collect_list('str_agegender').alias('list_agegender'))\
                        .select(['target', 'list_agegender'])
sdf_behavioral_prep = sdf_behavioral\
						.withColumn('str_behavioral', F.format_string('(%s, %s)', F.col('cohort_caption'), F.col('probability')))\
						.select(['target', 'str_behavioral'])\
						.groupBy('target')\
						.agg(F.collect_list('str_behavioral').alias('list_behavioral'))\
                        .select(['target', 'list_behavioral'])
sdf_llm_prompt_prep = sdf_agegender_prep\
						.join(sdf_behavioral_prep, on='target', how='inner')\
						.select(['target', 'list_agegender', 'list_behavioral'])

# メモリ管理
sdf_agegender.unpersist()
sdf_behavioral.unpersist()
sdf_agegender_prep.unpersist()
sdf_behavioral_prep.unpersist()
del sdf_agegender, sdf_behavioral
del sdf_agegender_prep, sdf_behavioral_prep
spark.catalog.clearCache()
gc.collect()

# Pandas UDF
@F.pandas_udf(types.StringType())
def generate_prompt_udf(pdf_agegender: pd.Series, pdf_behavioral: pd.Series) -> pd.Series:
	pdf_combined = pd.DataFrame({
						'list_agegender'  : pdf_agegender,
						'list_behavioral' : pdf_behavioral,
					})
	def _create_prompt(row):
		list_agegender  = row['list_agegender']
		list_behavioral = row['list_behavioral']
		str_agegender   = "[ " + ", ".join(list_agegender)  + " ]" if list_agegender  is not None else "[]"
		str_behavioral  = "[ " + ", ".join(list_behavioral) + " ]" if list_behavioral is not None else "[]"

		content = (
					"# 前提条件\n"
					"あなたは外資系コンサルティングファーム出身の、極めて優秀なプロダクトマーケティングマネージャー兼リテール戦略アナリストです。\n"
					"ある特定のターゲット地点（店舗やエリア）において、【特定の商品】を展開・販売するミッションを帯びています。\n"
					"この地点に訪れる顧客の「属性データ」と「行動（回遊）データ」を提供します。\n"
					"提供される「商品の詳細分析」「ターゲット地点の顧客属性データ」「顧客の回遊（行動）データ」を高度に統合・解釈し、データドリブンかつMECEな思考で、具体的な商戦・マーケティングのアクションプランを提案してください。\n\n"

					"# データの定義\n"
					"# 入力情報\n"
					"## 1.【販売するコア商材の分析情報】\n"
					f"- ブランド名: {BRAND_NAME}\n"
					
					"## 2.【属性データ】(targetの客層構成)\n"
					"- target         : 分析対象の場所\n"
					"- age            : 年齢層\n"
					"- gender         : 性別\n"
					"- probability    : targetにおけるその属性（年齢×性別）の構成比・出現確率\n\n"

					"## 3.【行動・回遊データ】(targetの顧客が他にどこに行っているか)\n"
					"- target         : 分析対象の場所\n"
					"- cohort_caption : targetに来訪した顧客が、他に訪れている別の場所・施設（競合や親和性の高いスポット）\n"
					"- probability    : targetの顧客が、そのcohort_captionを訪れる確率・親和度スコア\n\n"

					"# 実際の抽出データ\n"
					"【属性データ】\n"
					f"{str_agegender}\n\n"

					"【行動・回遊データ】\n"
					f"{str_behavioral}\n\n"

					"# 指示事項\n"
					"提供された3つのデータ（商品特性、顧客属性、回遊行動）を掛け合わせ、以下の構成でコンサルティングレポートを出力してください。各項目は抽象的な一般論を避け、データに基づく具体的な仮説と提案に落とし込んでください。\n\n"

					"## 1. 顧客ペルソナとカスタマージャーニーの深い洞察\n"
					"- 属性（age/gender）と回遊先（cohort_caption）の傾向から、ターゲット地点に来訪する「メイン客層の解像度の高いペルソナ（ライフスタイル、価値観）」を定義してください。\n"
					"- 彼らがどのような目的・文脈でターゲット地点とcohort_captionを行き来しているか、1日の行動導線（カスタマージャーニー）の仮説を立ててください。\n\n"

					"## 2. プロダクト・ロケーション・フィット（商材と場所の親和性）\n"
					"- 【コア商材の分析情報】と【地点の特性（顧客・回遊）】がどのようにマッチしているか（あるいはギャップがあるか）を分析してください。\n"
    				"- 顧客はなぜ、数ある場所の中で「このターゲット地点」で「この商品」を買うべきなのか、独自の提供価値（バリュープロポジション）を言語化してください。\n\n"

					"## 3. 具体的な商戦・マーケティング戦略への提言（ネクストアクション）\n"
					"- **店舗レイアウト・MD提案 :** 判明したペルソナと回遊目的を踏まえ、ターゲット地点でどのような商品展開、陳列、あるいは看板・POPのメッセージングを行うべきか。\n"
					"- **コラボ＆送客戦略      :** cohort_caption（他の来訪スポット）と競合して奪い合うべきか、あるいは相乗効果を狙って連携・クロスセルを図るべきか。具体的な施策案を提示してください。\n"
					"- **広告・販促アプローチ   :** この客層に対して、いつ、どのタイミング（例：cohort_captionに行く前/後など）で、どのようなプロモーションを打つのが最もCVR（コンバージョン率）が高いか提案してください。\n\n"

					"## 4. 期待される成果とモニタリング指標（KPI）\n"
					"- 提案した戦略を実行した場合に期待されるビジネスインパクトと、追跡すべき具体的なKPIを定義してレポートを締めくくってください。\n"
				)
		return content
	return pdf_combined.apply(_create_prompt, axis=1)

# LLM用プロンプトの生成
sdf_llm_prompt = sdf_llm_prompt_prep\
					.withColumn('prompt', generate_prompt_udf(col('list_agegender'), col('list_behavioral')))\
					.select(['target', 'prompt'])

# メモリ管理
sdf_llm_prompt_prep.unpersist()
del sdf_llm_prompt_prep
spark.catalog.clearCache()
gc.collect()

# 目標の場所毎にAPIを投げる
list_tasks = []
for row in tqdm(sdf_llm_prompt.toLocalIterator()):
	target_place  = row['target']
	target_prompt = row['prompt']

	async def _response(place:str, messages:list):
		res = await llmClient.complete(messages)
		return place, res

	messages  = []
	messages.append(SystemMessage(content=target_prompt))
	messages.append(UserMessage(content=product_analysis))
	list_tasks.append(_response(target_place, messages))

# メモリ管理
sdf_llm_prompt.unpersist()
del sdf_llm_prompt
spark.catalog.clearCache()
gc.collect()

# タスクの集約
analysis_datas = await asynctqdm.gather(*list_tasks)

# データフレームへの変換
pdf_final_analysis = pd.DataFrame([{
        'caption' : target_name,
        '分析結果'  : json.dumps(response_dict, indent=4, ensure_ascii=False),
    } for (target_name, response_dict) in analysis_datas])

# メモリ管理
del analysis_datas
gc.collect()

pdf_final_analysis

In [ ]:
TARGET = f"/Volumes/stgadintedmpadinteds/envdev/sandbox/Cohort_AI_Agent_Aiico/output/{BRAND_NAME}_analysis.parquet"
pdf_final_analysis.to_parquet(TARGET, compression="snappy")
